# Lab 2: Post-Training Quantization of a TensorFlow Model for Size Reduction

### Objective:
The goal of this lab is to apply post-training quantization to a pre-trained TensorFlow model, convert it to TFLite format, and evaluate the effect of quantization on model size and accuracy.

### Pre-requisites:
- Completion of Lab 1 or access to a trained TensorFlow model
- Understanding of TFLite conversion process
- TensorFlow, NumPy installed

In [7]:
# Step 1: Import Libraries
import tensorflow as tf
import numpy as np
from tensorflow import keras
import os

In [8]:
# Step 2: Load MNIST Dataset
mnist = keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_test = x_test / 255.0
x_test = x_test.astype(np.float32)

# Use a small test set for evaluation
x_test_sample = x_test[:100]
y_test_sample = y_test[:100]

In [9]:
# Step 3: Load Pre-trained Model (from Lab 1)
model = keras.models.load_model("mnist_model.keras")

In [10]:
# Step 4: Convert to TFLite with Post-Training Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Enable default post-training quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_quant_model = converter.convert()

# Save the quantized model
with open("mnist_model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

print("Quantized model saved as mnist_model_quant.tflite")

INFO:tensorflow:Assets written to: C:\Users\bhawa\AppData\Local\Temp\tmpo3gqxiz1\assets


INFO:tensorflow:Assets written to: C:\Users\bhawa\AppData\Local\Temp\tmpo3gqxiz1\assets


Saved artifact at 'C:\Users\bhawa\AppData\Local\Temp\tmpo3gqxiz1'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1886486148688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1886639445520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1886639445328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1886639445136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1886639447056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1886639444560: TensorSpec(shape=(), dtype=tf.resource, name=None)
Quantized model saved as mnist_model_quant.tflite


In [11]:
# Step 5: Evaluate the Quantized Model
interpreter = tf.lite.Interpreter(model_path="mnist_model_quant.tflite")
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

correct = 0
for i in range(len(x_test_sample)):
    input_data = np.expand_dims(x_test_sample[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    pred = np.argmax(output)
    if pred == y_test_sample[i]:
        correct += 1

accuracy = correct / len(x_test_sample)
print(f"Quantized Model Accuracy on 100 samples: {accuracy * 100:.2f}%")

Quantized Model Accuracy on 100 samples: 100.00%


C:\Users\bhawa\AppData\Roaming\Python\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [12]:
# Step 6: Compare File Sizes
original_model_size = os.path.getsize("mnist_model.keras") / 1024
tflite_model_size = os.path.getsize("mnist_model.tflite") / 1024
tflite_quantized_model_size = os.path.getsize("mnist_model_quant.tflite") / 1024

print(f"Original Model Size: {original_model_size:.2f} KB")
print(f"TFLite Model Size: {tflite_model_size:.2f} KB")
print(f"Quantized TFLite Model Size: {tflite_quantized_model_size:.2f} KB")
print(f"Compression Ratio: {original_model_size / tflite_quantized_model_size:.2f}x")

Original Model Size: 1308.86 KB
TFLite Model Size: 429.57 KB
Quantized TFLite Model Size: 113.91 KB
Compression Ratio: 11.49x
